# Cloud & Azure Foundations

Azure is a platform of about two hundred services running across roughly sixty regions worldwide. To learn it well, you do not start with the services — you start with the *shape of the platform itself*: how cloud computing differs from running your own servers, how Microsoft splits responsibility with you, how every resource you create slots into a hierarchy that controls billing and access, and how that hierarchy spans Microsoft's global network of regions and zones.

Get this skeleton right and every later service — VMs, storage accounts, databases, AKS clusters — drops cleanly onto it.

## Cloud service models

The three classic models are IaaS, PaaS, and SaaS, and they trade control for convenience.

**IaaS — Infrastructure as a Service.** You rent virtual hardware and run everything above it yourself. In Azure that is `Virtual Machines`, `Virtual Networks`, and managed disks. You pick the OS, patch it, install your runtime, manage the firewall. You are buying flexibility — anything you can run on a server, you can run here. AWS equivalent: EC2, VPC, EBS.

**PaaS — Platform as a Service.** You hand over the OS and the runtime; you bring code and config. `App Service` runs your web app, `Azure Functions` runs event handlers, `Azure SQL Database` runs the database engine. Microsoft patches the host, scales the underlying compute, handles failover. You give up the ability to SSH into a box; you get back hours per week. AWS equivalent: Elastic Beanstalk, Lambda, RDS.

**SaaS — Software as a Service.** Pure consumption: Microsoft 365, Dynamics 365, GitHub itself. You do not see infrastructure; you sign in. AWS does not have a strong SaaS portfolio — Microsoft does, and that historical strength shapes how Azure markets itself.

The practical rule when designing a system: **start with the most managed option that fits**. Reach for IaaS only when PaaS cannot give you the control you need. Every step down the stack is a step toward more knobs and more two-a-m pages.

## Shared responsibility

Cloud security is not *Microsoft handles it* — it is a split that **shifts depending on the service model**. In IaaS, you own everything inside the VM: the OS, the patches, the application, the data, the identity used to log in. Microsoft owns the physical host, the hypervisor, the network fabric, the building. In PaaS, Microsoft absorbs the OS and runtime layer; you still own your code, your data, and your identity configuration. In SaaS, almost everything is Microsoft's responsibility except your data and your users.

```
                      IaaS      PaaS      SaaS
  Data & access       you       you       you
  Application         you       you       MS
  Runtime / OS        you       MS        MS
  Virtualization      MS        MS        MS
  Physical host       MS        MS        MS
```

The two things that are **always yours**, every model, every service: your **data** and the **identities** you grant access to it. They never leave your side regardless of how managed the platform becomes. That is why identity (Entra ID) and data classification land first in any Azure security review.

## Azure Resource Manager — the control plane

Every action you take in Azure — clicking in the portal, running the `az` CLI, calling a Python SDK, applying a Bicep template — funnels through one API: **Azure Resource Manager**, or ARM. ARM is the *single entry point* that authenticates the request, checks RBAC, validates against Azure Policy, and finally tells the resource provider (Compute, Storage, Network, etc.) to do the work.

A few consequences fall out of having one control plane:

- **Anything you can do in the portal, you can do via API.** No portal-only features. The Activity Log captures every ARM call regardless of source — portal, CLI, SDK, or template — so you get a complete audit trail for free.
- **Templates are first-class.** Because the API speaks declarative JSON, you can describe an entire environment as a file (ARM template or Bicep) and apply it idempotently.
- **RBAC and Policy intercept every call** before the resource provider sees it. That makes governance enforceable, not advisory.

ARM is the analogue of AWS's CloudFormation, IAM evaluator, and API surface rolled into one. The cleaner abstraction is one of Azure's quiet strengths.

## The resource hierarchy

Every resource in Azure lives in a four-level hierarchy. Memorise this shape — RBAC, Policy, billing, and cost reporting all hang off it.

```
Tenant  (Entra ID directory — identity boundary)
 └── Management Group         (governance grouping, up to 6 levels)
      └── Subscription         (billing + quota boundary)
           └── Resource Group  (lifecycle + RBAC scope)
                └── Resource   (VM, storage account, database, ...)
```

**Tenant.** Your Entra ID directory. One identity boundary. Users, groups, and service principals live here. A tenant can hold many subscriptions; a subscription belongs to exactly one tenant.

**Management Group.** A tree above subscriptions used to apply Policy and RBAC at scale. The root MG is implicit; you build the tree below it (e.g. `Prod`, `NonProd`, `Sandbox`). Up to six levels deep. A policy assigned at an MG cascades to every subscription and resource under it.

**Subscription.** The **unit of billing and the unit of quota**. Spend caps, payment methods, and Azure resource limits (cores per region, public IPs per subscription) all live here. A small org might use one subscription; a large enterprise commonly carves out a subscription per workload, environment, or business unit to isolate blast radius and cost.

**Resource Group.** A logical container for resources that share a lifecycle. Deleting a resource group deletes everything in it — that is the feature, not a bug, because it makes teardown trivial. The RG is also a default RBAC scope and a metadata bucket via tags. A resource lives in exactly one RG; you cannot nest RGs.

**Resource.** The actual VM, storage account, key vault, etc. Every resource has a resource ID that fully encodes the hierarchy:

```
/subscriptions/{sub-id}/resourceGroups/{rg}/providers/{namespace}/{type}/{name}
```

The AWS comparison is loose. AWS has Organizations → OUs → Accounts → Resources, where the **account** is the strong isolation boundary. In Azure the **subscription** plays a similar role, but resource groups have no AWS analogue — they are a lightweight lifecycle grouping you will use constantly.

In [ ]:
# Walk the hierarchy with Azure CLI — assumes `az login` already done.

# 1. List management groups visible to you.
az account management-group list --query "[].{name:name, displayName:displayName}" -o table

# 2. List subscriptions and note which tenant each belongs to.
az account list --query "[].{name:name, id:id, tenant:tenantId, state:state}" -o table

# 3. Pick a subscription and create a resource group in it.
az account set --subscription "<sub-id>"
az group create --name rg-foundations-demo --location eastus --tags env=demo owner=ganesh

# 4. Drop a resource into it and inspect the fully-qualified resource ID.
az storage account create \
  --name stfoundationsdemo$RANDOM \
  --resource-group rg-foundations-demo \
  --location eastus \
  --sku Standard_LRS \
  --query id -o tsv
# /subscriptions/<sub-id>/resourceGroups/rg-foundations-demo/providers/Microsoft.Storage/storageAccounts/<name>

# 5. Tear the whole thing down in one shot — every resource in the RG goes with it.
az group delete --name rg-foundations-demo --yes --no-wait

## Regions, geographies, and availability zones

Azure runs in over **sixty regions** worldwide, grouped into **geographies** (e.g. United States, Europe, India) that respect data residency boundaries. A region is a cluster of datacenters within a metro area, connected by low-latency fibre, that presents itself to you as a single deployment target like `eastus` or `westeurope`.

Inside a region, Microsoft offers two failure-isolation primitives that often get confused.

**Availability Zones** are *physically separate datacenters within one region*, each with independent power, cooling, and network. Three zones per AZ-enabled region. A zone-redundant service — Standard Load Balancer, zone-redundant storage (ZRS), a zonal VM Scale Set — tolerates a full datacenter failure transparently. Not every region has zones yet; check the docs when picking one. AWS equivalent: AZs work the same way.

**Region pairs** are Microsoft's older HA primitive: each region is paired with another region in the same geography (e.g. `eastus` with `westus`, `northeurope` with `westeurope`) for cross-region replication features like geo-redundant storage and paired-region maintenance — Microsoft never patches both halves of a pair simultaneously. Newer regions and Azure's strategic direction favour Availability Zones, but pairs still underpin specific services and remain exam-relevant.

```
  Region: East US                        Region: West US
  +--------------------------+          +--------------------------+
  |  AZ 1    AZ 2    AZ 3    |   pair   |  AZ 1    AZ 2    AZ 3    |
  | [ dc ]  [ dc ]  [ dc ]   | <------> | [ dc ]  [ dc ]  [ dc ]   |
  +--------------------------+          +--------------------------+
        (within-region HA)                  (cross-region DR / GRS)
```

Rule of thumb: use Availability Zones for high availability inside a region; use a region pair (or another distant region) for disaster recovery across regions.

## Sovereign clouds

For workloads bound by regulation, Azure runs **isolated cloud instances** with separate fabric, separate identity, and separate compliance certifications:

- **Azure Government** — US federal, state, local, and Department of Defense workloads. Operated by US-screened personnel in physically segregated datacenters.
- **Azure China** — operated by **21Vianet** under Chinese law. A legally separate cloud, not just a region; Microsoft does not run it directly.
- *Azure Germany has been retired; you may still see legacy references in older docs.*

Sovereign clouds are **not transparent extensions of public Azure**. They have their own portal URLs, their own service availability matrices (some services lag or never arrive), and their own Entra ID tenants. Pick them only when you have a regulatory reason; otherwise stay in public Azure.

## Edge and hybrid extensions

Beyond the main regions, Microsoft pushes compute closer to users, devices, and on-prem datacenters with a family of edge and hybrid offerings. You will rarely deploy these unless you have a specific reason, but you should recognise the names:

- **Azure Edge Zones** — small footprints in metro areas, some carrier-hosted, for sub-ten-millisecond latency to end users. Analogue of AWS Local Zones and Wavelength.
- **Azure Stack Hub** — a full Azure-consistent cloud you run inside your own datacenter, for disconnected or sovereign scenarios.
- **Azure Stack HCI** — hyperconverged infrastructure for hybrid VM and AKS workloads on-prem.
- **Azure Stack Edge** — an appliance you put at a site for local inference and data preprocessing before forwarding to the cloud.
- **Azure Arc** — the reverse direction: bring on-prem servers, Kubernetes clusters, and SQL instances *under ARM's control plane* so you can govern them with the same RBAC, Policy, and Monitor you use for native Azure resources.

Arc is the strategically important one. Anything Arc-enabled shows up in your subscription, gets tagged, gets policy-enforced, and gets billed through your Enterprise Agreement. It is how Microsoft pitches a coherent hybrid story that AWS Outposts approaches only from the hardware side.

## Choosing a region

When you spin up your first resource, the region prompt is one of the first decisions, and it is harder to undo than it looks. Four factors drive the choice:

1. **Latency to your users.** Pick the region geographically closest to where most traffic originates. Speed-test probes can settle close calls.
2. **Data residency and compliance.** If regulation pins data to a country, the geography you pick has to satisfy that — and you may need a sovereign cloud.
3. **Service and SKU availability.** Not every region has every service or every SKU. Newer SKUs — the latest VM family, Premium SSD v2, Cosmos DB multi-region writes — light up unevenly. Always confirm against the *Products available by region* page before committing.
4. **Price.** Region prices differ. The same VM in `centralindia` is cheaper than in `westus`. If latency and compliance don't force your hand, price can tilt the choice.

Once you commit, **migrating regions is non-trivial** for stateful workloads: storage, databases, and identity flows are all region-pinned in subtle ways. Treat the first region pick as a multi-year decision, not a default.